# Fig. 3 — Export panels a–h in manuscript order

Run the first two setup cells, then run the desired panels.
The panels use fixed styles. Set their sizes with `PANEL_X_SIZE_MM`.
The panels read the saved human validation and dynamics results. Each exports as SVG, PNG and PDF under `outputs/figure3_main/` with a name from `fig3_a` to `fig3_h`.
Panels c–h show program weights, gene contributions, functional enrichment, TCGA survival, anti-PD-1 Cox results and longitudinal SW-CD8, respectively.


In [ ]:
from pathlib import Path
import importlib.util, json, os, sys
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm, Normalize

REPO = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p/'human_validation/figure_workbench').is_dir())
WB = REPO/'human_validation/figure_workbench'
HUMAN_RESULTS = Path(os.environ.get('HUMAN_VALIDATION_RESULTS', REPO/'human_validation'))
DYNAMICS_RESULTS = Path(os.environ.get('DYNAMICS_RESULTS', REPO/'dynamics_analysis/outputs'))
# Physical sizes are millimetres; fonts are points. Change these explicitly.
STYLE = dict(font_family='Arial', font_size=8, title_size=7.5, label_size=8,
             tick_size=7.5, legend_size=7.5, annotation_size=7.5,
             panel_label_size=11, font_scale=1.0, line_width=.6,
             png_dpi=600, pdf_dpi=600, formats=('pdf','svg','png'))
PROGRAMS = ['Precursor','Activation','Early effector','Cytotoxic','Dysfunction']
PROGRAM_LABELS = dict(zip(PROGRAMS,[r'$P_{\mathrm{pre}}$',r'$A_{\mathrm{act}}$',r'$E_{\mathrm{early}}$',r'$Y_{\mathrm{cyto}}$',r'$D_{\mathrm{dysf}}$']))
PROGRAM_COLORS = dict(zip(PROGRAMS,['#586B9A','#49A09D','#DA9B4F','#C75666','#9465A6']))
PROGRAM_COMPONENTS = ['weight_Precursor','weight_Activation','weight_Early_effector',
                      'weight_Cytotoxic','weight_Dysfunction']
LIBRARIES = ['GO Biological Process','Reactome','MSigDB Hallmark']

def apply_style(s=None):
    s = {**STYLE, **(s or {})}
    mpl.rcParams.update({'font.family':s['font_family'],'font.size':s['font_size'],
      'axes.titlesize':s['title_size'],'axes.labelsize':s['label_size'],
      'xtick.labelsize':s['tick_size'],'ytick.labelsize':s['tick_size'],
      'legend.fontsize':s['legend_size'],'axes.linewidth':s['line_width'],
      'axes.spines.top':False,'axes.spines.right':False,'xtick.major.size':2,
      'ytick.major.size':2,'pdf.fonttype':42,'ps.fonttype':42,'svg.fonttype':'path'})
    return s

def read(relative):
    relative = Path(relative)
    if relative.parts[0] == "human_validation":
        p = HUMAN_RESULTS.joinpath(*relative.parts[1:])
    elif relative.parts[:2] == ("dynamics_analysis", "outputs"):
        p = DYNAMICS_RESULTS.joinpath(*relative.parts[2:])
    else:
        p = REPO / relative
    return pd.read_csv(p)

def export(fig, stem, outdir, style=None):
    s = {**STYLE, **(style or {})}; outdir=Path(outdir); outdir.mkdir(parents=True,exist_ok=True)
    # font_scale changes every explicit text size, including annotations.
    if not getattr(fig,'_fig3_fonts_scaled',False):
        for text in fig.findobj(mpl.text.Text):
            text.set_fontfamily(s['font_family'])
            text.set_fontsize(text.get_fontsize()*s['font_scale'])
        fig._fig3_fonts_scaled=True
    # Automatic layout fills the requested canvas; do not crop/change its physical size.
    # Keep SVG lettering as paths even if an imported plotting helper resets rcParams.
    with mpl.rc_context({'svg.fonttype':'path'}):
        for fmt in s['formats']:
            fig.savefig(outdir/f'{stem}.{fmt}',dpi=s['png_dpi'] if fmt=='png' else s['pdf_dpi'],
                        facecolor='white',bbox_inches=None)
    print(stem)


apply_style()


In [ ]:
OUT = WB/'outputs/figure3_main'

# Global analysis/display settings. Each drawing cell sets only its output size.
PANEL_STYLE = {key:dict(STYLE) for key in 'abcdefgh'}
N_GENES_PER_SIGN = 8  # ranked solely by abs(frozen gene coefficient)
tr=['Control','Anti-PD1','Anti-4-1BB','Combination']
internal=['Control','Anti-PD1','Anti-41BB','Anti-PD1+Anti-41BB']

def standalone(key,draw,size_mm,stem=None):
    """Render one standalone figure with automatic layout at the requested size."""
    s=apply_style(PANEL_STYLE[key]); w,h=size_mm
    fig=plt.figure(figsize=(w/25.4,h/25.4),facecolor='white',layout='constrained')
    draw(fig,s)
    export(fig,stem or 'fig3_'+key,OUT,s)
    plt.show(); plt.close(fig)

# Curated functional themes, selected for interpretation after inspecting the
# full results, not a preregistered/significance-ranking rule. Both signs use
# the same eight rows; full-library adjusted P values remain unchanged.
MECHANISTIC_PATHWAYS = [
 ('MSigDB Hallmark','IL-2/STAT5 Signaling','IL-2/STAT5 signalling','Cytokine response and lymphocyte activation'),
 ('MSigDB Hallmark','Interferon Gamma Response','Interferon-γ response','Interferon-response gene composition'),
 ('MSigDB Hallmark','TNF-alpha Signaling via NF-kB','TNF–NF-κB signalling','Inflammatory response signalling'),
 ('MSigDB Hallmark','IL-6/JAK/STAT3 Signaling','IL-6/JAK/STAT3 signalling','Cytokine-associated state regulation'),
 ('GO Biological Process','T Cell Activation (GO:0042110)','T-cell activation','T-cell activation annotation'),
 ('GO Biological Process','Regulation of Natural Killer Cell Mediated Cytotoxicity (GO:0042269)','NK-cell cytotoxicity\nregulation','Cytotoxic-receptor regulation; retain one KLRC-related term'),
 ('Reactome','RUNX3 Regulates Immune Response and Cell Migration','RUNX3: immune response\nand migration','Immune regulation and migration; two negative overlap genes'),
 ('MSigDB Hallmark','Hypoxia','Hypoxia','Stress-associated gene composition'),
]
ENRICHMENT_STYLE=dict(bubble_base=8,bubble_scale=400,fdr_color_max=30,
                      dot_edge_width=.45,label_size=6.2)


## a — Longitudinal pseudocells (PNG, SVG and PDF)


In [ ]:
# Panel a: set the final width and height in millimetres.
PANEL_A_SIZE_MM = (90,75)

def draw_a(fig,s):
    # a: empirical robust-coverage ellipse in the saved diffusion coordinates.
    src=REPO/'dynamics_analysis/src/figure_helpers.py'
    spec=importlib.util.spec_from_file_location('cd8score_saved_vis',src)
    mod=importlib.util.module_from_spec(spec); sys.modules[spec.name]=mod; spec.loader.exec_module(mod)
    pc=read('dynamics_analysis/outputs/pseudocell/tables/diffmap_sim_day1_6_pseudocells.csv.gz')
    ax=fig.subplots()
    colors=['#555555','#3F7CAC','#E07A2D','#7B4F9D']
    ellipse_bounds=[]
    for g,label,color,marker in zip(internal,tr,colors,['o','^','s','D']):
        p=pc.loc[pc.treatment.eq(g)]
        ax.scatter(p.DC1,p.DC2,s=3.8,c=color,marker=marker,alpha=.65,linewidths=0,rasterized=True,label=label.replace("Anti-PD1", "anti-PD-1"))
        centers=p.groupby('time')[['DC1','DC2']].mean().to_numpy()
        bounds=mod.add_robust_coverage_ellipse(ax,p.DC1,p.DC2,color,required_points=centers)
        if bounds is not None:
            ellipse_bounds.append(bounds)
    limits=mod.calculate_longitudinal_ellipse_limits(pc,ellipse_bounds)
    ax.set(xlim=limits[0],ylim=limits[1])
    ax.set_xlabel('DC1',fontsize=8)
    ax.set_ylabel('DC2',fontsize=8)
    ax.tick_params(axis='both',labelsize=8)
    ax.legend(frameon=False,markerscale=2,loc='upper right',handletextpad=.2,labelspacing=.25,fontsize=8)

standalone('a',draw_a,PANEL_A_SIZE_MM)


## b — Direct neural drift field (PNG, SVG and PDF)


In [ ]:
# Panel b: set the final width and height in millimetres.
PANEL_B_SIZE_MM = (120,75)

def draw_b(fig,s):
    # b: frozen direct neural drift, shared color normalization and display field.
    bg=read('dynamics_analysis/outputs/pseudocell/tables/diffmap_sim_day1_6_pseudocells.csv.gz')
    bg['treatment']=bg.treatment.replace(dict(zip(internal,tr)))
    field=read('dynamics_analysis/outputs/vtheta/plot_data/vtheta_dense_field.csv.gz')
    meta=json.loads((DYNAMICS_RESULTS/'vtheta/source_data/pc_time_display_parameters.json').read_text())
    lo,hi=np.quantile(bg.Eff_Cytox_zstd,[.02,.98]); norm=TwoSlopeNorm(0,min(lo,-1e-12),max(hi,1e-12))
    axes=fig.subplots(2,2,sharex=True,sharey=True)
    for i,g in enumerate(tr):
        ax=axes.flat[i]
        p=bg.loc[bg.treatment.eq(g)]
        dots=ax.scatter(p.DC1,p.DC2,c=p.Eff_Cytox_zstd,cmap='RdBu_r',norm=norm,s=3.3,alpha=.52,linewidths=0,rasterized=True)
        f=field.loc[field.treatment.eq(g)].sort_values(['y','x']); n=int(round(np.sqrt(len(f))))
        ax.streamplot(np.sort(f.x.unique()),np.sort(f.y.unique()),
           np.ma.masked_invalid(f.display_U.to_numpy().reshape(n,n)),
           np.ma.masked_invalid(f.display_V.to_numpy().reshape(n,n)),
           color='#35434F',density=1.05,linewidth=.45,arrowsize=.5,minlength=.025,maxlength=2.6)
        ax.set(xlim=meta['main_limits'][0],ylim=meta['main_limits'][1])
        ax.set_title(g.replace("Anti-PD1", "anti-PD-1"),fontsize=8)
        ax.tick_params(axis='both',labelsize=8)
        if i>=2: ax.set_xlabel('DC1',labelpad=1,fontsize=8)
        else: ax.tick_params(labelbottom=False)
        if i%2==0: ax.set_ylabel('DC2',labelpad=1,fontsize=8)
        else: ax.tick_params(labelleft=False)
    cb=fig.colorbar(dots,ax=axes.ravel().tolist(),extend='both')

    vmin,vmax=norm.vmin,norm.vmax
    ticks=np.r_[np.linspace(vmin,0,3)[:-1],np.linspace(0,vmax,4)]
    cb.set_ticks(ticks)
    cb.ax.set_yticklabels([f'{x:.1f}' for x in ticks],fontsize=8)

    cb.set_label('Cytotoxic score',labelpad=1,fontsize=8)
    cb.ax.tick_params(labelsize=8)

standalone('b',draw_b,PANEL_B_SIZE_MM)


## c — Treatment-specific program weights (PNG, SVG and PDF)


In [ ]:
# Panel c: set the final width and height in millimetres.
PANEL_C_DELTA_SD = 0.5
PANEL_C_SIZE_MM = (65,60)  # Programme-weight panel dimensions are set below.

def draw_c(fig,s):
    all_alpha=read('dynamics_analysis/outputs/robustness/dynamic_importance_by_delta.csv')
    alpha=all_alpha.loc[np.isclose(all_alpha.delta_sd,PANEL_C_DELTA_SD)].copy()
    index=pd.MultiIndex.from_product([tr,PROGRAMS],names=['treatment','program'])
    alpha=alpha.set_index(['treatment','program']).loc[index].reset_index()
    matrix=alpha.pivot(index='program',columns='treatment',values='alpha_mean').loc[PROGRAMS,tr]
    # One shared, symmetric scale; no row-wise rescaling or pooling of deltas.
    vmax=np.ceil(np.abs(matrix.to_numpy()).max()/0.05)*0.05
    norm=TwoSlopeNorm(vmin=-vmax,vcenter=0,vmax=vmax)
    ax=fig.subplots()
    im=ax.imshow(matrix.to_numpy(),cmap='RdBu_r',norm=norm,aspect='auto',interpolation='nearest')
    ax.set(xticks=np.arange(len(tr)),xticklabels=[label.replace("Anti-PD1", "anti-PD-1") for label in tr],
           yticks=np.arange(len(PROGRAMS)),yticklabels=[PROGRAM_LABELS[p] for p in PROGRAMS])
    ax.tick_params(axis='both',length=0,pad=3,labelsize=8)
    ax.tick_params(axis='y',labelsize=8)
    plt.setp(ax.get_xticklabels(),rotation=30,ha='right',rotation_mode='anchor',fontsize=6)
    for row in range(len(PROGRAMS)):
        for col in range(len(tr)):
            value=matrix.iloc[row,col]
            ax.text(col,row,f'{value:.3f}',ha='center',va='center',fontsize=6,
                    color='white' if abs(value)>0.6*vmax else '#222222')
    for spine in ax.spines.values(): spine.set_visible(False)
    cb=fig.colorbar(im,ax=ax,ticks=[-vmax,0,vmax])
    cb.ax.set_title(r'$\alpha_j$',fontsize=8,pad=3)
    cb.ax.tick_params(labelsize=8,length=2,pad=2)
    cb.outline.set_visible(False)

standalone('c',draw_c,PANEL_C_SIZE_MM)


## d — Gene contributions to SW-CD8 (PNG, SVG and PDF)


In [ ]:
# Panel d: set the final width and height in millimetres.
PANEL_D_SIZE_MM = (75,68)  # Gene-contribution panel dimensions are set below.

def draw_d(fig,s):
    from matplotlib.lines import Line2D
    from matplotlib.patches import Patch
    weights=read('human_validation/TCGA_SKCM/gene_level_cd8score/cd8score_gene_weights.csv')
    weights=weights.loc[weights.variable_expression_all_tumors].copy()
    weights['absolute_weight']=weights.cd8score_gene_weight.abs()
    order=['absolute_weight','human_ensembl_gene']
    pos=weights.loc[weights.cd8score_gene_weight.gt(0)].sort_values(order,ascending=[False,True]).head(N_GENES_PER_SIGN)
    neg=weights.loc[weights.cd8score_gene_weight.lt(0)].sort_values(order,ascending=[False,True]).head(N_GENES_PER_SIGN)
    selected=pd.concat([pos,neg],ignore_index=True)
    ax=fig.subplots(); y=np.arange(len(selected))
    left_pos=np.zeros(len(y)); left_neg=np.zeros(len(y))
    for program,column in zip(PROGRAMS,PROGRAM_COMPONENTS):
        values=selected[column].to_numpy(); left=np.where(values>=0,left_pos,left_neg)
        ax.barh(y,values,left=left,height=.72,color=PROGRAM_COLORS[program],edgecolor='none',lw=0)
        left_pos+=np.maximum(values,0); left_neg+=np.minimum(values,0)
    ax.scatter(selected.cd8score_gene_weight,y,marker='|',s=18,lw=.8,color='#111111',zorder=5)
    ax.axvline(0,color='.5',lw=.5); ax.axhline(N_GENES_PER_SIGN-.5,color='.8',lw=.4)
    ax.set(yticks=y,yticklabels=selected.human_symbol,ylim=(len(y)-.4,-.6),
           xlim=(-0.01, 0.03), xticks=[-0.01, 0, 0.01, 0.02, 0.03])
    ax.tick_params(axis='y',length=0,labelsize=8)
    ax.tick_params(axis='x',labelsize=8)
    ax.set_xlabel(r'Gene coefficient $\beta_g$',fontsize=8,labelpad=2)
    handles=[Line2D([],[],color='#111111',marker='|',linestyle='none',
                    markersize=6,markeredgewidth=.8,label=r'Net $\beta_g$')]
    handles += [Patch(facecolor=PROGRAM_COLORS[p],label=PROGRAM_LABELS[p]) for p in PROGRAMS]
    ax.legend(handles=handles, loc='lower right', bbox_to_anchor=(1.1, 0),
          ncol=2, frameon=False, fontsize=8,
          handlelength=1.2, handletextpad=.4, columnspacing=.3,
          labelspacing=.55, borderaxespad=.5)

standalone('d',draw_d,PANEL_D_SIZE_MM)


## e — Functional enrichment of positively and negatively weighted genes (PNG, SVG and PDF)


In [ ]:
# Panel e: set the final width and height in millimetres.
PANEL_E_SIZE_MM = (70,70)

def draw_e(fig,s):
    e=ENRICHMENT_STYLE
    base='human_validation/TCGA_SKCM/gene_level_cd8score/all_cd8score_enrichment/'
    tables={sign:read(base+sign+'_weight_enrichment_results.csv') for sign in ['positive','negative']}
    rows=[]
    for rank,(library,term,label,rationale) in enumerate(MECHANISTIC_PATHWAYS):
        for sign in ['positive','negative']:
            frame=tables[sign]
            selected=frame.loc[frame.library.eq(library)&frame.term.str.strip().eq(term)]
            row=selected.iloc[0].to_dict()
            row.update(display_order=rank,display_label=label,selection_rationale=rationale,weight_sign=sign)
            rows.append(row)
    table=pd.DataFrame(rows)
    norm=Normalize(0,e['fdr_color_max']); cmap=plt.get_cmap('viridis')


    # Separate plot/legend rows; constrained layout determines their positions.
    plot_area,legend_area=fig.subfigures(2,1,height_ratios=[4,1.3])
    ax=plot_area.subplots()
    for y,idx in enumerate(range(len(MECHANISTIC_PATHWAYS))):
        for x,sign in enumerate(['positive','negative']):
            r=table.loc[table.display_order.eq(idx)&table.weight_sign.eq(sign)].iloc[0]
            if r.overlap_count==0:
                ax.text(x,y,'–',ha='center',va='center',fontsize=8,color='.55')
            else:
                color=cmap(norm(-np.log10(max(r.bh_fdr,np.finfo(float).tiny))))
                ax.scatter(x,y,s=e['bubble_base']+e['bubble_scale']*r.gene_ratio,
                           facecolor=color if r.bh_fdr<.05 else 'white',
                           edgecolor='#555555',lw=e['dot_edge_width'])
    ax.set(xlim=(-.55,1.55),ylim=(len(MECHANISTIC_PATHWAYS)-.5,-.5),xticks=[0,1],
           xticklabels=[f'{label}\n(n={int(tables[sign].query_gene_count.iloc[0])})'
                        for sign,label in [('positive','Positive'),('negative','Negative')]],
           yticks=np.arange(len(MECHANISTIC_PATHWAYS)),
           yticklabels=[x[2] for x in MECHANISTIC_PATHWAYS])
    ax.xaxis.tick_top(); ax.tick_params(axis='x',length=0,labelsize=8.5,pad=2)
    ax.tick_params(axis='y',length=0,labelsize=8,pad=2)
    for spine in ax.spines.values(): spine.set_visible(False)

    from matplotlib.lines import Line2D
    handles=[Line2D([],[],linestyle='none',marker='o',
                    markersize=np.sqrt(e['bubble_base']+e['bubble_scale']*ratio),
                    markerfacecolor='#AAAAAA',markeredgecolor='#555555',
                    markeredgewidth=.35,label=f'{ratio:.0%}') for ratio in [.02,.08,.15]]
    # Shared footer: size legend above; two-line note left of the colorbar.
    footer=legend_area.add_gridspec(2,2,width_ratios=[1,1.25],height_ratios=[1,1.2])
    legend_ax=legend_area.add_subplot(footer[0,:]); legend_ax.axis('off')
    ratio_title=Line2D([],[],linestyle='none',label='Gene ratio:')
    legend_ax.legend(handles=[ratio_title,*handles],loc='center',ncol=4,
                     frameon=False,fontsize=8,handlelength=.8,
                     handletextpad=.5,columnspacing=1)
    note_ax=legend_area.add_subplot(footer[1,0]); note_ax.axis('off')
    note_ax.text(.5,.5,'–: no overlap',
                 ha='center',va='center',fontsize=8)
    colorbar_ax=legend_area.add_subplot(footer[1,1])
    colorbar_ax.set_box_aspect(.05)
    cb=legend_area.colorbar(mpl.cm.ScalarMappable(norm=norm,cmap=cmap),
                            cax=colorbar_ax,orientation='horizontal')
    cb.set_label('−log10(BH-FDR)',fontsize=8,labelpad=2)
    cb.ax.xaxis.set_label_position('top')
    cb.set_ticks([0,10,20,30]); cb.ax.tick_params(labelsize=8,length=1,pad=1)


standalone('e',draw_e,PANEL_E_SIZE_MM)


## f — TCGA survival by score group (PNG, SVG and PDF)


In [ ]:
# Panel f: set the final width and height in millimetres.
PANEL_F_SIZE_MM = (70,55)

def draw_f(fig,s):
    # f: exact saved Kaplan-Meier steps, no survival refit.
    curves=read('human_validation/TCGA_SKCM/outputs/km_curve_source_data.csv')
    stats=read('human_validation/TCGA_SKCM/outputs/km_median_split_results.csv')
    curves=curves.loc[curves.cohort.eq('All tumors') & curves.score.eq('CD8score_Control')]
    stats=stats.loc[stats.cohort.eq('All tumors') & stats.score.eq('CD8score_Control')]
    ax=fig.subplots()
    # for group,color in [('High','#2475A9'),('Low','#C77B34')]:
    #     p=curves.loc[curves.group.eq(group)].sort_values('time_days'); n=int(stats.loc[stats.group.eq(group),'n'].iloc[0])
    #     ax.step(p.time_days/365.25,p.survival,where='post',lw=1,color=color,label=f'{group} (n={n})')
    for group,color in [('High','#2475A9'),('Low','#C77B34')]:
        p=curves.loc[curves.group.eq(group)].sort_values('time_days')
        n=int(stats.loc[stats.group.eq(group),'n'].iloc[0])

        name = {'High': 'High SW-CD8', 'Low': 'Low SW-CD8'}[group]
        ax.step(p.time_days/365.25,p.survival,
                where='post',lw=1,color=color,
                label=f'{name} (n={n})')        
    ax.set(ylim=(0,1.02))
    ax.set_xlabel('Time (years)',fontsize=8)
    ax.set_ylabel('Overall survival',fontsize=8)
    ax.tick_params(axis='both',labelsize=8)
    ax.set_title('TCGA-SKCM dataset',pad=1,fontsize=8)
    ax.legend(frameon=False,loc='upper right',fontsize=8,handlelength=1.2)
    ax.text(.03,.08,f"P = {stats.logrank_p_value.iloc[0]:.2g}",transform=ax.transAxes,fontsize=8)
    ax.xaxis.labelpad=1; ax.yaxis.labelpad=1

standalone('f',draw_f,PANEL_F_SIZE_MM)


## g — Cox analysis of PFS and OS in the anti-PD-1 cohort (PNG, SVG and PDF)


In [ ]:
# Panel g: set the final width and height in millimetres.
PANEL_G_SIZE_MM = (70,55)

def draw_g(fig,s):
    # g: Efron Cox estimates and 95% confidence intervals from saved results.
    cox=read('human_validation/Liu2019/outputs/cox_continuous_results.csv'); cox=cox.loc[cox.score.eq('CD8score_PD1')].set_index('endpoint').loc[['PFS','OS']]
    ax=fig.subplots()
    for y,endpoint in [(1,'PFS'),(0,'OS')]:
        p=cox.loc[endpoint]; hr=p.hazard_ratio_per_score_sd
        ax.errorbar(hr,y,xerr=[[hr-p.hazard_ratio_ci95_low],[p.hazard_ratio_ci95_high-hr]],fmt='o',ms=3,color='#356A96',capsize=2,lw=1)
        ax.text(.46,y+.22,f'HR {hr:.2f}; P = {p.wald_p_value:.3g}',fontsize=8)
    ax.axvline(1,color='.5',ls='--',lw=.6)
    ax.set(yticks=[1,0],yticklabels=['PFS','OS'],xlim=(.45,1.15),ylim=(-.55,1.6))
    ax.set_xlabel('HR per 1 SD (95% CI)',fontsize=8)
    ax.tick_params(axis='both',labelsize=8)
    ax.set_title('anti-PD-1 cohort (n=104)',pad=1,fontsize=8)
    ax.set_xticks([.5,.75,1]); ax.xaxis.labelpad=1

standalone('g',draw_g,PANEL_G_SIZE_MM)


## h — Longitudinal changes in mouse and human SW-CD8 (PNG, SVG and PDF)


In [ ]:
# Panel h: set the final width and height in millimetres.
PANEL_H_SIZE_MM = (70,55)

def draw_h(fig,s):
    # h: saved mouse and human score changes on a normalized stage axis.
    base='human_validation/GSE272993_longitudinal/results/cd8score_longitudinal_validation/'
    mouse=read(base+'mouse_pd1_specific_delta_cd8score_summary.csv'); mouse=mouse.loc[mouse.time.between(1,8)].sort_values('time')
    human=read(base+'human_delta_cd8score_summary.csv').set_index('timepoint').loc[['Baseline','Follow Up 1','Follow Up 2','Follow Up 3','Follow Up 4']].reset_index()
    ax=fig.subplots()
    ax.plot((mouse.time-1)/7,mouse['mean'],color='#2475A9',lw=1,label='Mouse PD1-specific')
    ax.errorbar(np.arange(5)/4,human['mean'],yerr=human['sem'],color='#CF842B',marker='o',ms=2.4,lw=.9,capsize=1.7,label='Human (4 patients)')
    ax.axhline(0,color='.5',ls='--',lw=.5)
    ax.set_xlabel('Normalized stage',fontsize=8)
    ax.set_ylabel(r'Baseline-relative $\Delta$SW-CD8',fontsize=8)
    ax.tick_params(axis='both',labelsize=8)
    ax.set_title('Early score change',pad=1,fontsize=8)
    ax.legend(frameon=False,loc='upper left',fontsize=8,handlelength=1.2)
    ax.set_xticks([0,.5,1]);ax.set_yticks([0,0.05,0.1,0.15]); ax.xaxis.labelpad=1; ax.yaxis.labelpad=1

standalone('h',draw_h,PANEL_H_SIZE_MM)
